# PySpark Hello World

This notebook demonstrates basic PySpark operations: creating data, saving to files (CSV, JSON, Parquet), loading data, manipulation, and analysis.

In [ ]:
import os
import sys

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, desc

# 1. Initialize Spark Session
# We set master to local[*] to use all available cores on your machine
spark = SparkSession.builder \
    .appName("PySparkHelloWorld") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

### 2. Create Dummy Data and Generate Files
We will create a DataFrame in memory and save it as CSV, JSON, and Parquet to satisfy your file requirements.

In [ ]:
# Create dummy data
data = [
    ("Alice", "Engineering", 75000, 28, "F"),
    ("Bob", "Marketing", 60000, 35, "M"),
    ("Charlie", "Engineering", 80000, 32, "M"),
    ("David", "HR", 55000, 29, "M"),
    ("Eve", "Marketing", 62000, 40, "F"),
    ("Frank", "Sales", 70000, 45, "M"),
    ("Grace", "Engineering", 90000, 38, "F")
]
columns = ["Name", "Department", "Salary", "Age", "Gender"]

df_initial = spark.createDataFrame(data, columns)
print("Initial Data:")
df_initial.show()

In [ ]:
# Define output paths (saving inside a 'data' folder)
csv_path = "data/employees_csv"
json_path = "data/employees_json"
parquet_path = "data/employees_parquet"

# Save as CSV (with header)
df_initial.write.mode("overwrite").option("header", "true").csv(csv_path)

# Save as JSON
df_initial.write.mode("overwrite").json(json_path)

# Save as Parquet (Compressed by default, efficient)
df_initial.write.mode("overwrite").parquet(parquet_path)

print(f"Files created in {os.getcwd()}/data/")

### 3. Load Data
Now we load the data back from the Parquet file (usually the most efficient format for Spark).

In [ ]:
df = spark.read.parquet(parquet_path)
df.printSchema()
df.show()

### 4. Data Manipulation
Filter rows (Age > 30) and add a new column (10% Salary Bonus).

In [ ]:
# Filter: Age > 30
df_filtered = df.filter(col("Age") > 30)

# Manipulation: Add 10% bonus column
df_bonus = df_filtered.withColumn("Bonus", col("Salary") * 0.10)

df_bonus.show()

### 5. Data Analysis
Group by Department and calculate average Salary and count of employees.

In [ ]:
analysis_df = df.groupBy("Department").agg(
    count("*").alias("Employee_Count"),
    avg("Salary").alias("Avg_Salary"),
    avg("Age").alias("Avg_Age")
).orderBy(desc("Avg_Salary"))

analysis_df.show()